## CREDIT SCORING MODEL

### 1. IMPORT LIBRARIES

In [ ]:
import pandas as pd
import numpy as np

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Preprocessing
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split

# Models
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

In [ ]:
# Evaluation Metrics
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report,
    RocCurveDisplay
)

### 2. LOAD DATASET

In [ ]:
df = pd.read_csv('/content/german_credit_data.csv',delimiter=',')


# Display first 5 rows
print(df.head())

### 3. DATA UNDERSTANDING

In [ ]:
print("\nDataset Shape:")
print(df.shape)

print("\nColumn Names:")
print(df.columns)

print("\nDataset Information:")
print(df.info())

print("\nMissing Values:")
print(df.isnull().sum())

### 4. HANDLE MISSING VALUES

In [ ]:
# Fill categorical missing values with mode
for col in df.select_dtypes(include='object').columns:
    df[col].fillna(df[col].mode()[0], inplace=True)

# Fill numerical missing values with median
for col in df.select_dtypes(include=['int64', 'float64']).columns:
    df[col].fillna(df[col].median(), inplace=True)

### 5. FEATURE ENGINEERING

In [ ]:
# Creating Age Group Feature

df['Age_Group'] = pd.cut(
    df['Age'],
    bins=[18, 30, 45, 60, 100],
    labels=['Young', 'Adult', 'Middle_Age', 'Senior']
)

# Loan Amount Per Duration Feature

df['Loan_Per_Month'] = df['Credit amount'] / df['Duration']

### 6. ENCODE CATEGORICAL VARIABLES

In [ ]:
label_encoder = LabelEncoder()

for col in df.select_dtypes(include='object').columns:
    df[col] = label_encoder.fit_transform(df[col])

# Encode newly created Age_Group column
df['Age_Group'] = label_encoder.fit_transform(df['Age_Group'])

### 7. DEFINE FEATURES AND TARGET

In [ ]:
# Define features and target using the 'Loan_Risk' column
target_column = 'Loan_Risk'

X = df.drop(['Loan_Per_Month', target_column], axis=1) # Drop original continuous and the new target
y = df[target_column]

print(f"Target Column: {target_column}")
print("Distribution of target variable:")
print(y.value_counts())

### 8. FEATURE SCALING

In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

### 9. TRAIN-TEST SPLIT (Re-run with new target)

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y # Stratify to maintain class balance in train/test sets
)

### 10. TRAIN MODELS (Re-run with new target)

In [ ]:
# Logistic Regression
lr_model = LogisticRegression()
lr_model.fit(X_train, y_train)

# Decision Tree
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)

# Random Forest
rf_model = RandomForestClassifier(random_state=42)
rf_model.fit(X_train, y_train)

### 11. MODEL EVALUATION FUNCTION

In [ ]:
def evaluate_model(model, X_test, y_test, model_name):

    y_pred = model.predict(X_test)

    # Probability prediction for ROC-AUC
    y_prob = model.predict_proba(X_test)[:, 1]

    accuracy = accuracy_score(y_test, y_pred)
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    roc_auc = roc_auc_score(y_test, y_prob)

    print(f"\n================ {model_name} ================")
    print(f"Accuracy  : {accuracy:.4f}")
    print(f"Precision : {precision:.4f}")
    print(f"Recall    : {recall:.4f}")
    print(f"F1-Score  : {f1:.4f}")
    print(f"ROC-AUC   : {roc_auc:.4f}")

    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

    # Confusion Matrix
    cm = confusion_matrix(y_test, y_pred)

    plt.figure(figsize=(7, 6))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
    plt.title(f'{model_name} - Confusion Matrix')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # ROC Curve
    fig_roc, ax_roc = plt.subplots(figsize=(7, 6))
    RocCurveDisplay.from_estimator(model, X_test, y_test, ax=ax_roc)
    plt.title(f'{model_name} - ROC Curve')
    plt.tight_layout()
    plt.show()

    return {
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1,
        'ROC-AUC': roc_auc
    }

### 12. EVALUATE ALL MODELS

In [ ]:
print("Logistic Regression")
lr_metrics = evaluate_model(lr_model, X_test, y_test, "Logistic Regression")

In [ ]:
print("Decision Tree")
dt_metrics = evaluate_model(dt_model, X_test, y_test, "Decision Tree")

In [ ]:
print("Random Forest")
rf_metrics = evaluate_model(rf_model, X_test, y_test, "Random Forest")

### 13. FEATURE IMPORTANCE (RANDOM FOREST)

In [ ]:
feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': rf_model.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
)

print("\nTop Important Features:")
print(feature_importance.head(10))

# Plot Feature Importance

plt.figure(figsize=(10,6))

sns.barplot(
    x='Importance',
    y='Feature',
    data=feature_importance.head(10)
)

plt.title('Top 10 Important Features')
plt.show()

### 14. FINAL RESULT

In [ ]:
print("\nProject Completed Successfully!")
print("Models Used:")
print("1. Logistic Regression")
print(f"   - Accuracy: {lr_metrics['Accuracy']:.4f}, Precision: {lr_metrics['Precision']:.4f}, Recall: {lr_metrics['Recall']:.4f}, F1-Score: {lr_metrics['F1-Score']:.4f}, ROC-AUC: {lr_metrics['ROC-AUC']:.4f}")
print("2. Decision Tree")
print(f"   - Accuracy: {dt_metrics['Accuracy']:.4f}, Precision: {dt_metrics['Precision']:.4f}, Recall: {dt_metrics['Recall']:.4f}, F1-Score: {dt_metrics['F1-Score']:.4f}, ROC-AUC: {dt_metrics['ROC-AUC']:.4f}")
print("3. Random Forest")
print(f"   - Accuracy: {rf_metrics['Accuracy']:.4f}, Precision: {rf_metrics['Precision']:.4f}, Recall: {rf_metrics['Recall']:.4f}, F1-Score: {rf_metrics['F1-Score']:.4f}, ROC-AUC: {rf_metrics['ROC-AUC']:.4f}")

print("\nEvaluation Metrics Used:")
print("- Accuracy")
print("- Precision")
print("- Recall")
print("- F1-Score")
print("- ROC-AUC")